# Build `all_macro_data.parquet` from `all_macro_data_long.csv`

Converts the long-format CSV into the compact Parquet store that `build_dashboard.py` reads
(ZSTD-compressed, dictionary-encoded - roughly 45x smaller and much faster to load).

Assumes **`all_macro_data_long.csv`** is in this same folder. **Run All Cells.**

In [ ]:
# ============================================================
#  Setup - locate the CSV and check deps
# ============================================================
import sys, subprocess, time
from pathlib import Path

def _find_dir():
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "all_macro_data_long.csv").exists():
            return d
    raise FileNotFoundError(
        "all_macro_data_long.csv not found. Put this notebook in the same folder "
        "as all_macro_data_long.csv and re-run."
    )

HERE    = _find_dir()
CSV     = HERE / "all_macro_data_long.csv"
PARQUET = HERE / "all_macro_data.parquet"
print("Folder  :", HERE)
print("CSV     :", CSV.name, "(" + str(round(CSV.stat().st_size / 1e6, 1)) + " MB)")
print("Output  :", PARQUET.name)

def _missing(m):
    try:
        __import__(m); return False
    except Exception:
        return True
need = [m for m in ("pandas", "pyarrow") if _missing(m)]
if need:
    print("Installing missing deps:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *need], check=True)
print("Deps    : pandas + pyarrow ok")

In [ ]:
# ============================================================
#  Convert  CSV  ->  Parquet
# ============================================================
import pandas as pd
t0 = time.time()

# 1) read the long-format CSV (columns: date, category, indicator, ticker, field, value)
df = pd.read_csv(CSV)
print("read", f"{len(df):,}", "rows | columns:", list(df.columns))

# 2) parse the date column; the four label columns stay as strings (pyarrow
#    dictionary-encodes repetitive strings automatically) and value stays float.
df["date"] = pd.to_datetime(df["date"])

# 3) write the compact store the dashboard reads (ZSTD compression, no index)
df.to_parquet(PARQUET, engine="pyarrow", compression="zstd", index=False)
print("wrote", PARQUET.name, "in", round(time.time() - t0, 1), "s")

In [ ]:
# ============================================================
#  Verify
# ============================================================
import pyarrow.parquet as pq
back = pd.read_parquet(PARQUET)
csv_mb = CSV.stat().st_size / 1e6
pq_mb  = PARQUET.stat().st_size / 1e6

print("rows   :", f"{len(back):,}", "(CSV had", f"{len(df):,})", "->", "OK" if len(back) == len(df) else "MISMATCH")
print("size   :", round(csv_mb, 1), "MB CSV  ->", round(pq_mb, 2), "MB parquet  (" + str(round(csv_mb / pq_mb, 1)) + "x smaller)")
print("tickers:", back["ticker"].nunique(), "| indicators:", back["indicator"].nunique(), "| categories:", back["category"].nunique())
print("dates  :", str(back["date"].min().date()), "->", str(back["date"].max().date()))
print()
print("schema:")
print(pq.ParquetFile(PARQUET).schema)

# Bloomberg ticker quirks (double spaces / literal %) must survive the round-trip
tk = set(back["ticker"])
quirks = [t for t in ("IP  YOY Index", "IP  CHNG Index", "CLEV16%M", "CHALYOY% Index") if t in tk]
print()
print("quirk tickers preserved:", quirks if quirks else "(none of the sample quirks in this data)")
print("Done - all_macro_data.parquet is ready for build_dashboard.py / build_dashboard.ipynb")